# Assignment 2
*Group 3: Fernando Green, Tyler Hensel, Jacob Ramer, Elise Wallace*

## Goal of the Exercise

It is important to keep the overarching goal of this exercise, which is to calculate the safety stock by SKU, where the inventory type is a finished good (FG) and the stocking type is made-to-stock (MTS).

It is also important to notate the formula for calculating safety stock, which is below:

$$ SS = z_{\alpha} \times \sigma_{d} \times \sqrt{LT}$$

Knowing this, we know that the key features of this dataset would include the SKU, stocking type, quantity, and lead time.

## Import Packages and Dataset
Since we ended up using the SciPy.Stat ppf function, we went ahead and imported the stats package from scipy.

In [176]:
import pandas as pd
import numpy as np
from plotnine import *
import scipy.stats as stats
from scipy.stats import norm

In [177]:
transaction_data = pd.read_csv("../data/transaction_data.csv")
meta_data = pd.read_csv("../data/meta_data.csv")

## Question 1 — Required Transformation
Create a transformed dataset to filter for appropriate SKUs that are also aggregated with the following statistics: minimum, maximum, average, median, variance, and standard deviation of quantity, along with average lead time.

### Answer 1:
#### Data Cleansing
Similar to assignment 1, we want to clean the data by updating the headers to match the metadata, dropping any NaN values for SKU number, omit the transactions where we the order quantity is less than or equal to 0, omit any transactions where the unit price is less than or equal to 0, and remove any transactions with extreme outliers (3.5-7.7 years).

In this case, we only want the data to show where the stocking type is MTS. Since we already know from Assignment 1 that we would not fill in the NaN values, we can just filter for any MTS products. 

In [178]:
transaction_title = ["sku_number", "inventory_type", "stocking_type", "lead_time", "unit_price", "manufacturing_site", "division_code", "transaction_date", "order_quantity"]
transaction_data.columns = transaction_title

In [179]:
lt_3std = 3*np.std(transaction_data["lead_time"])
lt_mean = np.mean(transaction_data["lead_time"])

transaction_data_clean = transaction_data.query('order_quantity >= 0 and unit_price >= 0 and abs(lead_time - @lt_mean) <= @lt_3std')
transaction_data_clean = transaction_data_clean.query('stocking_type == "MTS"')
transaction_data_clean = transaction_data_clean[transaction_data_clean["sku_number"].notna()]

transaction_data_clean.describe()

,lead_time,unit_price,order_quantity
count,422650.000000,422650.000000,422650.000000
mean,26.329904,842.034967,85.046107
std,4.162539,340.854922,37.282082
min,14.000000,34.391014,0.000000
25%,28.000000,986.262413,73.000000
50%,28.000000,997.739775,95.000000
75%,28.000000,1004.995525,110.000000
max,28.000000,1026.135082,198.000000


#### Data Transformation
In order to use aggregate the quantity by SKU, we have to create a new dataframe. We will use the "groupby" function, along with the aggregate function. Since we are trying to aggregate multiple summarizations of order_quantity, we will have to call the specific functions we'd like in brackets (Pandas development team, n.d.).

Because of the way we are aggregating the quantity and lead time by SKU, it creates multi-level headers. We will overwrite this by just manually renaming the column headers.

##### Aggregating Order Quantities and Lead Times

In [180]:
transformed_data = (transaction_data_clean
    .groupby("sku_number")
    .aggregate({"order_quantity":["min", "max", "mean", "median", "var", "std"],
                "lead_time":"mean"})
    .reset_index())

transformed_data

sku_number order_quantity                                                  \
                          min  max        mean median          var        std   
0     00478A8F              0  166   83.887398   94.0  1384.699204  37.211547   
1     00874431              0  164   98.990621  100.0   517.469276  22.747951   
2     018AA89B              0   21    8.207547    6.0    37.244557   6.102832   
3     01FAD69F              0   23    8.810345    6.5    44.647610   6.681887   
4     02452E5E              0   87   22.218563   19.0   282.081216  16.795274   
..         ...            ...  ...         ...    ...          ...        ...   
491   FED6CA5E             37  165   98.974586  100.0   375.621471  19.380956   
492   FEE42AC0              0   28    9.652174    8.0    46.994885   6.855282   
493   FF42B9D4              0   29   10.881356   10.0    56.140853   7.492720   
494   FF711BAA             18  174  100.761988  101.0   397.543668  19.938497   
495   FF9E44A8              1   21    9.555556    9.0    28.297980   5.319585   

     lead_time  
          mean  
0    26.128414  
1    27.778429  
2    14.000000  
3    14.000000  
4    19.805389  
..         ...  
491  28.000000  
492  14.000000  
493  14.000000  
494  28.000000  
495  14.000000  

[496 rows x 8 columns]

##### Overwriting Column Names

In [181]:
transformed_data.columns = ["sku_number",
                           "min_order_quantity",
                           "max_order_quantity",
                           "avg_order_quantity",
                           "median_order_quantity",
                           "var_order_quantity",
                           "std_order_quantity",
                           "avg_lead_time"]
transformed_data.sample(5)

,sku_number,min_order_quantity,max_order_quantity,avg_order_quantity,median_order_quantity,var_order_quantity,std_order_quantity,avg_lead_time
333,AA75B32D,0,161,83.497605,94.0,1393.105888,37.324334,26.125958
105,3C765470,0,70,22.763473,19.5,271.814758,16.486806,19.889222
78,2EB05945,0,35,10.686275,8.0,77.779608,8.819275,14.000000
445,E0C722AA,0,26,9.008264,9.0,36.441598,6.036688,14.000000
245,7C0EB55C,0,177,98.739118,100.0,548.605354,23.422326,27.748824


## Question 2 — Safety Stock Calculation
Use the transformed data from Question 1 to calculate three safety stock values for each SKU for the service level coefficients of 75%, 90%, and 95%. You will need to find the z-score of the given service level coefficients, for which you can use the [ppf](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.rv_continuous.ppf.html) function of SciPy.Stat module of the [SciPy](https://docs.scipy.org/doc/scipy/reference/) package, or simply use the numeric values from any standard normal distribution function.

### Answer 2:

#### Type of Distribution
##### Using Coefficient of Variation Rule for Normal Distribution
First, we want to check to see if our inventory is normally distributed or if a different distribution should be used. We will add another column for the coefficient of variation. For formula for the coefficient of variation is as follows:

$\text{CV} = \frac{\text{Std Dev}}{\text{Mean}}$

If the CV is less thna 30%, demand is relatively stable and predictable, relative to its volume, indicating that we can most likely use a normal distribution (Oracle, n.d.).

Now, we calculate the CV and find the SKUs which we can apply a normal distribution.

In [182]:
transformed_data["cv"] = transformed_data["std_order_quantity"]/transformed_data["avg_order_quantity"]

In [183]:
transformed_data.query('cv <= 0.3')

,sku_number,min_order_quantity,max_order_quantity,avg_order_quantity,median_order_quantity,var_order_quantity,std_order_quantity,avg_lead_time,cv
1,00874431,0,164,98.990621,100.0,517.469276,22.747951,27.778429,0.229799
8,048B94FE,19,165,99.866627,100.0,418.892640,20.466867,28.000000,0.204942
18,0839933A,27,169,99.800364,100.0,408.627395,20.214534,28.000000,0.202550
22,0B213A3E,0,174,95.973356,99.0,748.468811,27.358158,27.317460,0.285060
25,0D4A17C7,0,161,96.637068,99.0,630.134749,25.102485,27.541120,0.259760
...,...,...,...,...,...,...,...,...,...
479,F633F4BB,36,171,100.611180,100.5,370.224114,19.241209,28.000000,0.191243
485,FA9DDBC6,0,164,97.201572,99.0,632.874514,25.156997,27.551937,0.258813
486,FB796951,0,164,96.745396,99.0,642.765586,25.352822,27.532182,0.262057
491,FED6CA5E,37,165,98.974586,100.0,375.621471,19.380956,28.000000,0.195818


##### Variance to Mean Ratio
Next, we will use a variance to mean ratio to determine whether we the SKUs have a binomial, poisson, or negative binomial distribution, using different "generalized" thresholds (CASRAI, n.d.).

$\text{VMR} = \frac{\text{Variance}}{\text{Mean}}$

In [184]:
transformed_data["VMR"] = transformed_data["var_order_quantity"]/transformed_data["avg_order_quantity"]

##### Filtering for Binomial Distributions
After filtering for any SKUs with a high coefficient of variation and variance to mean ratio much then 1, we identified 0 SKUs which indicate a binomial distribution.

In [185]:
transformed_data.query('cv > 0.3 and VMR < 0.8')

,sku_number,min_order_quantity,max_order_quantity,avg_order_quantity,median_order_quantity,var_order_quantity,std_order_quantity,avg_lead_time,cv,VMR


##### Filtering for Poisson Distributions
After filtering for any SKUs with a high coefficient of variation and medium variance to mean ratio close to 1, we identified 0 SKUs which indicate a poisson distribution. We selected a ratio between 0.8 and 1.2

In [186]:
transformed_data.query('cv > 0.3 and VMR >= 0.8 and VMR <1.2')

,sku_number,min_order_quantity,max_order_quantity,avg_order_quantity,median_order_quantity,var_order_quantity,std_order_quantity,avg_lead_time,cv,VMR


##### Filtering for Negative Binomial Distributions
After filtering for any SKUs with a high coefficient of variation and a variance to mean ratio greater than 1, we identified the remaining non-normally distributed SKUs as having a negative binomial distribution. This meeans that most SKUs have a fairly low demand/order quantity but experience erratic or unpredictable high-quantity spikes. Using a safety stock calculation which assumes a normal distribution, we may experience more stockouts on these SKUs.

In [187]:
transformed_data.query('cv > 0.3 and VMR >= 1.2')

,sku_number,min_order_quantity,max_order_quantity,avg_order_quantity,median_order_quantity,var_order_quantity,std_order_quantity,avg_lead_time,cv,VMR
0,00478A8F,0,166,83.887398,94.0,1384.699204,37.211547,26.128414,0.443589,16.506641
2,018AA89B,0,21,8.207547,6.0,37.244557,6.102832,14.000000,0.743563,4.537843
3,01FAD69F,0,23,8.810345,6.5,44.647610,6.681887,14.000000,0.758414,5.067635
4,02452E5E,0,87,22.218563,19.0,282.081216,16.795274,19.805389,0.755912,12.695745
5,02CFF6A9,0,88,24.605442,22.0,254.567342,15.955167,21.000000,0.648441,10.345977
...,...,...,...,...,...,...,...,...,...,...
489,FDDDA064,0,194,82.669574,94.0,1514.247527,38.913334,27.172481,0.470709,18.316866
490,FE1EE752,0,43,10.421053,9.0,74.564509,8.635074,14.000000,0.828618,7.155180
492,FEE42AC0,0,28,9.652174,8.0,46.994885,6.855282,14.000000,0.710232,4.868839
493,FF42B9D4,0,29,10.881356,10.0,56.140853,7.492720,14.000000,0.688583,5.159362


#### Find z-scores for Percentiles for Normal Distributions

In [188]:
z_75 = norm.ppf(0.75)
z_90 = norm.ppf(0.90)
z_95 = norm.ppf(0.95)

#### Calculating Safety Stock for Normal Distributions
Now we use the following function to calculate the safety stock for each SKU. We will use our transformed dataset to complete this.

$$ SS = z_{\alpha} \times \sigma_{d} \times \sqrt{LT}$$

We also want to display whole numbers for these, so we will use the floor division (//) and remainder (%) functions to achieve this. The floor division will give us the "rounded" integer. We also use the remainder function, along with a logical operator to return a value of 0 or 1. If the remainder > 0, then the result should be 1. If the remainder is 0, then the result of that subcalculation would be 0. We also want to change the data type to "int64".

To demonstrate this, we've provided an example below:

- We select a random value of 17.4
- When we do the floor division, it will return a value of 17.
- When we do the remainder, it should return a value of 4.
- We also have a logical operator for the remainder, saying if the remainder (4) is greater than 0, then return a value of 1.
- In this case, since we have a remainder that is greater than 0, it would return a value of 1.
- We add this boolean response of 1 to the floor division result of 17.
- Our final result would be 18.

##### Safety Stock at 75%

In [189]:
raw_ss_75_norm = z_75 * transformed_data["std_order_quantity"] * np.sqrt(transformed_data["avg_lead_time"])
transformed_data["ss_75_norm"] = (raw_ss_75_norm // 1 + (raw_ss_75_norm % 1 > 0)).astype("int64")
transformed_data.sample(5)

,sku_number,min_order_quantity,max_order_quantity,avg_order_quantity,median_order_quantity,var_order_quantity,std_order_quantity,avg_lead_time,cv,VMR,ss_75_norm
301,96F77983,0,80,19.712042,16.0,237.041061,15.396138,21.219895,0.781052,12.025191,48
27,0FD5FE2C,0,76,22.991758,20.0,256.680373,16.021248,19.711538,0.696826,11.164017,48
61,243ECBFB,0,20,7.918367,8.0,27.951531,5.286921,14.000000,0.667678,3.529961,14
319,A2E0090F,23,170,99.584994,99.0,402.950250,20.073621,28.000000,0.201573,4.046295,72
146,4FCC9D48,0,165,88.898649,97.0,1232.602827,35.108444,26.275468,0.394927,13.865259,122


##### Safety Stock at 90%

In [190]:
raw_ss_90_norm = z_90 * transformed_data["std_order_quantity"] * np.sqrt(transformed_data["avg_lead_time"])
transformed_data["ss_90_norm"] = (raw_ss_90_norm // 1 + (raw_ss_90_norm % 1 > 0)).astype("int64")
transformed_data.sample(5)

,sku_number,min_order_quantity,max_order_quantity,avg_order_quantity,median_order_quantity,var_order_quantity,std_order_quantity,avg_lead_time,cv,VMR,ss_75_norm,ss_90_norm
365,BA3B7FF9,0,40,10.011494,8.0,50.184838,7.084126,14.000000,0.707599,5.012722,18,34
395,CA656E8B,0,79,21.972644,19.0,238.014493,15.427718,19.744681,0.702133,10.832310,47,88
248,7C7832C8,0,77,25.957875,23.5,252.018406,15.875088,21.000000,0.611571,9.708745,50,94
404,CEB08662,44,163,100.477015,101.0,384.677339,19.613193,28.000000,0.195201,3.828511,71,134
129,45E5F0E1,0,32,8.712963,7.5,45.234597,6.725667,14.000000,0.771915,5.191643,17,33


##### Safety Stock at 95%

In [191]:
raw_ss_95_norm = z_95 * transformed_data["std_order_quantity"] * np.sqrt(transformed_data["avg_lead_time"])
transformed_data["ss_95_norm"] = (raw_ss_95_norm // 1 + (raw_ss_95_norm % 1 > 0)).astype("int64")
transformed_data.sample(5)

,sku_number,min_order_quantity,max_order_quantity,avg_order_quantity,median_order_quantity,var_order_quantity,std_order_quantity,avg_lead_time,cv,VMR,ss_75_norm,ss_90_norm,ss_95_norm
303,9894C79B,38,157,100.247919,100.5,407.383473,20.183743,28.000000,0.201338,4.063760,73,137,176
124,41E4BEB8,0,28,10.303571,10.0,45.487987,6.744478,14.000000,0.654577,4.414779,18,33,42
376,C2D23FE2,0,173,97.185502,99.0,643.548094,25.368250,27.552511,0.261029,6.621853,90,171,220
473,F1032A80,0,32,10.267606,9.0,62.598793,7.911940,14.000000,0.770573,6.096727,20,38,49
188,61743189,0,31,10.882353,9.0,52.553117,7.249353,14.000000,0.666157,4.829205,19,35,45


#### Calculating Safety Stock for Negative Binomial Distributions
For the SKUs with a negative binomial distribution, we want to identify the reorder point and average demand during lead time to calculate safety stock. The formula is as follows:
$$\text{Safety Stock} = ROP - \mu_{LT}$$

To obtain the reorder point, we would use the SciPy.stats nbinom.ppf function.
To use the function, we would also need to calculate several other parameters. We would need the shape/dispersion (n), probability parameter (p), average demand during lead time, and average variance during lead time. The formulas are as follows:

$$n = \frac{\mu_{LT} \times p}{1 - p}$$
$$p = \frac{\mu_{LT}}{\sigma^2_{LT}}$$
$$\mu_{LT} = \mu_{\text{daily}} \times \text{Lead Time}$$
$$\sigma^2_{LT} = \sigma^2_{\text{daily}} \times \text{Lead Time}$$

The n parameter (sometimes known as k) and the p-parameter are from a statistical technique called "method of moments" (Zaiontz, n.d.).

In [192]:
transformed_data["mean_during_lt"] = transformed_data["avg_order_quantity"]*transformed_data["avg_lead_time"]
transformed_data["var_during_lt"] = transformed_data["var_order_quantity"]*transformed_data["avg_lead_time"]

In [193]:
transformed_data["p_parameter"] = transformed_data["mean_during_lt"] / transformed_data["var_during_lt"]

In [194]:
transformed_data["n_parameter"] = ((transformed_data["mean_during_lt"] * 
                                    transformed_data["p_parameter"]) / 
                                    (1- transformed_data["p_parameter"]))

In [195]:
transformed_data["rop_75"] = stats.nbinom.ppf(0.75, transformed_data["n_parameter"], transformed_data["p_parameter"])
transformed_data["rop_90"] = stats.nbinom.ppf(0.90, transformed_data["n_parameter"], transformed_data["p_parameter"])
transformed_data["rop_95"] = stats.nbinom.ppf(0.95, transformed_data["n_parameter"], transformed_data["p_parameter"])

##### Adding Safety Stock columns for negative binomial distributions
Now, we substrate the reorder point from the average demand during lead time to obtain the safety stock at 75%, 90%, and 95%.

We would use the same rounding methodology as the normal distribution safety stocks to ensure that we are providing whole values for safety stock quantities.

In [196]:
raw_ss_75_bn = transformed_data["rop_75"] - transformed_data["mean_during_lt"]
raw_ss_90_bn = transformed_data["rop_90"] - transformed_data["mean_during_lt"]
raw_ss_95_bn = transformed_data["rop_95"] - transformed_data["mean_during_lt"]

transformed_data["ss_75_bn"] = (raw_ss_75_bn // 1 + (raw_ss_75_bn % 1 > 0)).astype("int64")
transformed_data["ss_90_bn"] = (raw_ss_90_bn // 1 + (raw_ss_90_bn % 1 > 0)).astype("int64")
transformed_data["ss_95_bn"] = (raw_ss_95_bn // 1 + (raw_ss_95_bn % 1 > 0)).astype("int64")

#### Finalized Safety Stock Column
Since we want to just consolidate the "correct" safety stock to each SKU, we will use the np.where function to call upon the safety stock at 75^, 90%, and 95%, depending on the value of the coefficient of variation. Since we already identified that anything with a coefficient of variation which is greater than 30% is using a negative binomial distribution, we just need to build logic that says:

If the CV is less than or equal to 0.3, then grab the NORMAL safety stock, else use the binomial negative safety stock calculation.

In [202]:
transformed_data["ss_75"] = np.where(transformed_data["cv"] <= 0.3,
                                     transformed_data["ss_75_norm"],
                                     transformed_data["ss_75_bn"])

transformed_data["ss_90"] = np.where(transformed_data["cv"] <= 0.3,
                                     transformed_data["ss_90_norm"],
                                     transformed_data["ss_90_bn"])

transformed_data["ss_95"] = np.where(transformed_data["cv"] <= 0.3,
                                     transformed_data["ss_95_norm"],
                                     transformed_data["ss_95_bn"])

transformed_data.sample(5)

,sku_number,min_order_quantity,max_order_quantity,avg_order_quantity,median_order_quantity,var_order_quantity,std_order_quantity,avg_lead_time,cv,VMR,...,n_parameter,rop_75,rop_90,rop_95,ss_75_bn,ss_90_bn,ss_95_bn,ss_75,ss_90,ss_95
54,202A9197,0,83,21.361596,18.0,266.956421,16.338801,18.852868,0.764868,12.497026,...,35.028828,448.0,496.0,526.0,46,94,124,46,94,124
246,7C13009B,0,30,9.747899,8.0,47.868110,6.918678,14.000000,0.709761,4.910608,...,34.897539,153.0,170.0,181.0,17,34,45,17,34,45
304,989A2064,0,170,86.442523,95.0,1200.849366,34.653274,26.614954,0.400882,13.891882,...,178.458328,2419.0,2532.0,2602.0,119,232,302,119,232,302
462,E76592EF,0,28,10.133333,8.0,54.625989,7.390940,14.000000,0.729369,5.390723,...,32.310551,160.0,178.0,190.0,19,37,49,19,37,49
333,AA75B32D,0,161,83.497605,94.0,1393.105888,37.324334,26.125958,0.447011,16.684381,...,139.084537,2307.0,2429.0,2504.0,126,248,323,126,248,323


## Question 3 — Safety Stock Distribution
Now that we have the safety stock values, write appropriate code to answer the following questions. For this question, you may focus on the 95% service level coefficient.

- Which SKU has the largest safety stock?
- Which SKU has the smallest safety stock?
- What is the average safety stock value across all SKUs?

### Answer 3:
Before answering questions, now I want to create a copy of my transformed data into a neat and clean table with just the SKU and safety stocks for each method (since this would also be a way to validate the data). We create a new dataframe from the aggregated transformed data with sku number, and the 3 columns for safety stock at 95% with both methods (normal and negative binomial).

In [203]:
selected_columns = ["sku_number", "ss_95_norm", "ss_95_bn", "ss_95"]
data_95_ss = transformed_data[selected_columns].copy()

#### Largest Safety Stock
In order to find the SKU with the largest safety stock, using a 95% service level agreement (SLA), we would need to sort the values by DESCENDING values from the "ss_95" column. We also used the .head function, but wanted to print multiples, just in case there were multiple SKUs with the highest safety stock quantity.

The SKU with the largest safety stock is AC3ADABF.

In [204]:
data_95_ss.sort_values("ss_95", ascending = False).head(5)

,sku_number,ss_95_norm,ss_95_bn,ss_95
342,AC3ADABF,349,364,364
20,089FA02B,340,350,350
166,5978F66E,333,345,345
489,FDDDA064,334,344,344
471,F0610752,331,340,340


#### Smallest Safety Stock
In order to find the SKU with the smallest safety stock, using a 95% service level agreement (SLA), we would need to sort the values by ASCENDING values from the "ss_95" column. We also used the .head function, but wanted to print multiples, just in case there were multiple SKUs with the smallest safety stock quantity.

The SKU with the smallest safety stock is 58F77D5E.

In [200]:
data_95_ss.sort_values("ss_95", ascending = True).head(5)

,sku_number,ss_95_norm,ss_95_bn,ss_95
163,58F77D5E,30,31,31
61,243ECBFB,33,35,35
495,FF9E44A8,33,35,35
49,1F900448,34,35,35
85,2FA78943,37,38,38


#### Average Safety Stock
To obtain the average safety stock, we can just use the .describe() function to find the mean.

The average safety stock is about ~122 units.

In [201]:
data_95_ss.describe()

,ss_95_norm,ss_95_bn,ss_95
count,496.000000,496.000000,496.000000
mean,118.768145,122.292339,121.741935
std,89.431429,90.820287,90.359227
min,30.000000,31.000000,31.000000
25%,44.000000,47.000000,47.000000
50%,67.500000,70.500000,70.500000
75%,176.000000,178.000000,176.000000
max,349.000000,364.000000,364.000000


## References

CASRAI. (n.d.). Negative binomial regression overdispersion. CASRAI Guides. Retrieved November 12, 2026, from https://casrai.org/guides/negative-binomial-regression-overdispersion

Oracle. (n.d.). NetSuite help center: Inventory management guidelines. Oracle Help Center. Retrieved November 12, 2026, from https://docs.oracle.com/en/cloud/saas/netsuite/ns-online-help/article_0702084957.html#subsect_75201401147

Pandas development team. (n.d.). Pandas.DataFrame.aggregate. pandas. Retrieved November 7, 2026, from https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.aggregate.html

Zaiontz, C. (n.d.). Method of moments: Negative binomial distribution. Real Statistics Using Excel. Retrieved November 12, 2026, from https://real-statistics.com/distribution-fitting/method-of-moments/method-of-moments-negative-binomial-distribution/